# Бутстрэп-доверительные интервалы и OOB-оценка

Два независимых подтверждения устойчивости качества при малой выборке, на тюнингованных финалистах (логрег, лес, XGBoost, CatBoost на наборах significant и no_collinear), гиперпараметры из ноутбука 07.

Бутстрэп-ДИ: для каждого финалиста считаем ROC-AUC и PR-AUC на out-of-fold предсказаниях train (без утечки) с 95% доверительным интервалом по 2000 бутстрэп-повторам. Интервал показывает, насколько оценка устойчива к составу выборки.

OOB-AUC: для случайного леса есть встроенная оценка по наблюдениям, не попавшим в бутстрэп-подвыборку дерева. Это независимый от кросс-валидации взгляд на обобщение. У бустинга и логрега OOB в этом смысле нет, для них опираемся на бутстрэп-ДИ. Логика в `src/bootstrap_ci.py`.

In [ ]:
import sys, pathlib, warnings
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))
warnings.filterwarnings('ignore')

import pandas as pd
from IPython.display import display
from src import bootstrap_ci, config

table = bootstrap_ci.run()
print('\nБутстрэп-ДИ по финалистам:')
display(table)
print('OOB случайного леса:')
display(pd.read_csv(config.TABLES_DIR / 'oob_auc.csv'))

## Вывод

Бутстрэп-доверительные интервалы подтверждают устойчивость качества и согласуются с вложенной CV (ноутбук 07). Лучшая оценка у случайного леса на no_collinear: ROC-AUC 0.772 с 95% ДИ [0.710; 0.833]. CatBoost (0.764) и XGBoost (0.761) на том же наборе идут следом, на наборе significant все финалисты около 0.73. Доверительные интервалы у всех финалистов широкие (около ±0.06-0.07) и полностью перекрываются: при выборке в 218 пациентов модели статистически неразличимы, что мы уже видели на тесте и во вложенной CV.

PR-AUC лежит в диапазоне 0.50-0.55 при базовой линии случайного классификатора около 0.32 (распространенность рецидива), то есть модели несут реальный сигнал и для несбалансированной задачи, а не только по ROC-AUC.

OOB-оценка случайного леса (по наблюдениям вне бутстрэп-мешка дерева) дает 0.721 на significant и 0.749 на no_collinear, в пределах бутстрэп-ДИ и близко к OOF-оценке (0.772) и вложенной CV (0.775). Это независимый от кросс-валидации взгляд, и он подтверждает, что лес обобщает без переобучения: оценка по отложенным наблюдениям внутри бэггинга совпадает с честной кросс-валидацией.

Общий итог: качество финалистов устойчиво и воспроизводится тремя независимыми способами (вложенная CV, бутстрэп-ДИ, OOB), потолок около 0.77 по ROC-AUC задан объемом данных.